In [1]:
%pip install python-flint==0.9.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 50.8 MB/s  0:00:00


In [6]:
from google.colab import files
uploaded = files.upload()

Saving d170_factors.py to d170_factors.py


In [7]:
from d170_factors import factors, roots

print(len(factors["real"]))     # 134
print(len(factors["complex"]))  # 124

134
124


In [3]:
import math
from fractions import Fraction
from flint import arb, fmpq, ctx

ctx.dps = 650  # Arithmetic precision; separate from root precision.
d=170


def exact(x):
    """Convert an integer or rational to an exact FLINT rational."""
    if isinstance(x, float):
        raise TypeError("Use exact integers or rationals, not floats.")
    if isinstance(x, Fraction):
        return fmpq(x.numerator, x.denominator)
    return fmpq(x)


def func(poly):
    """Polynomial evaluation using Arb balls and Horner's method."""
    coefficients = [arb(exact(c)) for c in poly]

    def evaluate(x):
        x = arb(exact(x))
        value = arb(0)

        for c in reversed(coefficients):
            value = value*x + c

        return value

    return evaluate


def primitive(poly, d=170):
    """
    Return F(t) = W(t)*exp(-t), where

        W - W' = t^(d/2-1)*P(t).

    Consequently, the signed weighted integral from a to b
    equals F(a) - F(b).
    """
    if not isinstance(d, int) or d < 2 or d % 2:
        raise ValueError("d must be a positive even integer.")
    if len(poly) == 0:
        raise ValueError("Polynomial must not be empty.")

    # Exact coefficients of q(t) = t^(d/2-1)*P(t).
    q = [fmpq(0)]*(d//2 - 1) + [exact(c) for c in poly]

    # W = q + q' + q'' + ...
    # Compute it exactly using W_k = q_k + (k+1)*W_(k+1).
    W = q.copy()
    for k in range(len(q) - 2, -1, -1):
        W[k] = q[k] + (k + 1)*W[k + 1]

    evaluate_W = func(W)

    def F(x):
        return evaluate_W(x)*(-arb(exact(x))).exp()

    return F


def integ(poly, a, b, d=170):
    """
    Arb enclosure of the SIGNED integral

        integral_a^b P(t)*t^(d/2-1)*exp(-t) dt.

    Finite endpoints must be exact integers or rationals.
    """
    F = primitive(poly, d)

    if b == math.inf:
        return F(a)  # F(infinity) = 0 exactly.

    return F(a) - F(b)


def L1norm(poly, roots, d=170):
    """
    Enclose the radial L1 norm of

        f(x) = P(pi*|x|^2)*exp(-pi*|x|^2).

    roots contains ALL positive real roots of P as integer
    pairs (numerator, denominator). Extra nonpositive roots
    and repeated roots are allowed.
    """
    cuts = sorted({
        Fraction(p, q)
        for p, q in roots
        if Fraction(p, q) > 0
    })

    # Construct the antiderivative only once.
    F = primitive(poly, d)

    previous = F(0)
    value = arb(0)

    for root in cuts:
        current = F(root)
        value += abs(previous - current)
        previous = current

    # Final interval: last positive root to infinity.
    # This also handles a polynomial with no positive roots.
    value += abs(previous)

    # Radial change-of-variables normalization.
    return value / math.factorial(d//2 - 1)

In [17]:
import math
from flint import fmpq, fmpq_poly


def half_dimension(d):
    if isinstance(d, bool) or not isinstance(d, int) or d < 2 or d % 2:
        raise ValueError("d must be a positive even integer.")
    return int(d // 2)


def mult(poly1, poly2):
    """Exact polynomial multiplication; coefficients in ascending order."""
    P = fmpq_poly([exact(c) for c in poly1])
    Q = fmpq_poly([exact(c) for c in poly2])
    return (P * Q).coeffs()


def innerprod(poly1, poly2, d=170):
    """
    Exact inner product of
        poly1(pi|x|^2)*exp(-pi|x|^2)
    and
        poly2(pi|x|^2)*exp(-pi|x|^2).
    """
    a = half_dimension(d)
    product = mult(poly1, poly2)

    # Integral of t^(a+j-1)*exp(-2t) / Gamma(a).
    # For j=0 this is 2^(-a).
    moment = fmpq(1, 2**a)
    value = fmpq(0)

    for j, coefficient in enumerate(product):
        value += coefficient * moment
        moment *= fmpq(a + j, 2)

    return value


def laguerre_mod(n, d=170):
    """Coefficients of L_n^(d/2-1)(2t), UNNORMALIZED."""
    a = half_dimension(d)

    if isinstance(n, bool) or not isinstance(n, int) or n < 0:
        raise ValueError("n must be a nonnegative integer.")

    return [
        fmpq(
            (-2)**j * math.comb(n + a - 1, n - j),
            math.factorial(j)
        )
        for j in range(n + 1)
    ]


def l2normsquare_lag(n, d=170):
    """Exact squared L2 norm of the unnormalized Laguerre–Gaussian."""
    a = half_dimension(d)
    return fmpq(math.comb(n + a - 1, n), 2**a)


def l1norm_upper_lag(n, d=170):
    """Integer upper bound for its L1 norm."""
    a = half_dimension(d)
    return math.comb(n + a - 1, n) * 3**n


def Error(poly, d=170):
    """Exact rational upper bound for ||P_- f||_1."""
    a = half_dimension(d)
    poly = [exact(c) for c in poly]

    # Remove trailing zero coefficients.
    while poly and poly[-1] == 0:
        poly.pop()

    degree = len(poly) - 1
    error = fmpq(0)

    for n in range(1, degree + 1, 2):
        # Exact Laguerre coefficient:
        # c_n = <f, phi_n> / ||phi_n||_2^2.
        phi_n = laguerre_mod(n, d)
        coefficient = (innerprod(poly, phi_n, d) / l2normsquare_lag(n, d))

        error += abs(coefficient) * l1norm_upper_lag(n, d)

    return error

In [13]:
from flint import arb, ctx


def certified_ratio(poly, roots, d=170, digits=650):
    ctx.dps = digits
    a = half_dimension(d)
    if type(a)==float:
      raise ValueError('False!')

    # Exact inner product with the UNNORMALIZED Gaussian.
    I = innerprod(poly, [1], d)

    if I == 0:
        raise ValueError("The Gaussian pairing is zero.")

    # Squared pairing with the L2-normalized Gaussian:
    # |<f, g_d>|^2 = 2^(d/2) * I^2.
    # This remains an exact rational number.
    pairing_squared = 2**a * I**2

    # Ball enclosure for ||f||_1.
    norm_f = L1norm(poly, roots, d)

    # Exact rational upper bound for ||P_- f||_1.
    error = Error(poly, d)

    # Certified upper bound for ||P_+ f||_1.
    norm_plus_upper = (norm_f + arb(error)).upper()

    if not norm_plus_upper.is_finite() or norm_plus_upper <= 0:
        raise ArithmeticError("Invalid norm bound.")

    # A ball enclosing a LOWER-BOUND expression for the desired ratio.
    ratio = arb(pairing_squared) / norm_plus_upper**2

    if not ratio.is_finite() or ratio.lower() <= 0:
        raise ArithmeticError("Increase arithmetic precision.")

    root = ratio.root(d)

    return {
        "gaussian_pairing_squared": pairing_squared,
        "l1_f_enclosure": norm_f,
        "l1_minus_upper": arb(error).upper(),
        "l1_plus_upper": norm_plus_upper,
        "ratio_lower": ratio.lower(),
        "root_lower": root.lower(),
    }

In [20]:
from flint import fmpz_poly

def construct_polynomial(factors):
    P = fmpz_poly([1])

    # Real factors: D*t - A
    for D, A in factors["real"]:
        P *= fmpz_poly([-A, D])

    # Complex-pair factors: (s*t - a)^2 + b^2
    for s, a, b in factors["complex"]:
        P *= fmpz_poly([
            a*a + b*b,  # Constant coefficient
            -2*s*a,    # Coefficient of t
            s*s,       # Coefficient of t^2
        ])

    return P

P = construct_polynomial(factors)

# Ascending coefficient list for our earlier functions:
poly = P.coeffs()

print("Degree:", P.degree())  # 382

#the_roots=[fmpq(A,D) for D,A in factors["real"] if A>0]
the_roots=[(A,D) for D,A in factors["real"] if A>0]

print(certified_ratio(P, the_roots)["root_lower"])

Degree: 382
[0.77905217742973143089831957851739378372670940401747004001913248584559624331355254841730475289293683564918277506601473971161468127950868362121897928575227915135998994891050353045912862458787783565627097685179055063753008741818653194335690882566478259861422048125425642862124014276670368572356242390054552949158411133325046639933726109839146029599578261401170147371512101569685800269420863630003755751444935595088332891492104146250948444700541172489680127926388012656757905708758738427993742151346440221432816363307401140162934233884158182475012663988948899410236364750476538403382121827763907978130831711963223682861396199367070128614801458719896494073 +/- 2.17e-652]
